### 1. Save model tas series at obs locations:
1. Load subdaily data for each simulation
2. Gaussian blur each timestep
3. Calculate/store time series at each obs location

In [1]:
from Montreal_UHI_toolbox import *
import dask_image.ndfilters as ndfilters

/runoff/gulley/.miniconda3/lib/python3.12/site-packages/gribapi/__init__.py:23: UserWarning: ecCodes 2.39.0 or higher is recommended. You are running version 2.14.1
  warnings.warn(


In [ ]:
station_set = obs
# 1. Load subdaily (or daily tbh) data for each simulation
subdaily = {}
subdaily['tas_C'],subdaily['tas_T'] = get_outputs('tas')

# [LOAD MORE INTO SUBDAILY HERE AFTER]

In [3]:
# 2. Gaussian blur each timestep (ie no temporal blurring)

# Using the conventions from gaussian_blur_xarray
#sigma = (0.,1.5,1.5) # tuple (sigma_temporal,sigma_y,sigma_x)
blurred = {}

# Creating function to leverage dask
def gaussian(da,sigma=(0.,1.5,1.5)):
    return ndfilters.gaussian_filter(da,sigma=sigma)

# Using gaussian inside custom xarray ufunc  
for key in subdaily.keys():
    blurred[key] = xr.apply_ufunc(
        gaussian,
        subdaily[key],
        input_core_dims=[['time','rlat','rlon']],
        output_core_dims=[['time','rlat','rlon']],
        dask='allowed'
    )


In [4]:
# 3. Time series at each station_set location

series = {} # the dataset(s) to save

# Comparisons have to be made in rotated pole to extract time series data
station_set_rotated_points = rotated_pole.transform_points(ccrs.PlateCarree(), station_set['lon'].values, station_set['lat'].values)
station_set_rlon = station_set_rotated_points[:, 0]
station_set_rlat = station_set_rotated_points[:, 1]

# Interpolate station_set locations, grab time series from blurred fields
for key in subdaily.keys():
    # Select nearest point to station from blurred sim data
    series[key] = blurred[key].sel(rlat=xr.DataArray(station_set_rlat, dims='points'),rlon=xr.DataArray(station_set_rlon, dims='points'),method='nearest')


In [6]:
key = 'tas_C'
series[key].to_netcdf(f'/runoff/gulley/St_Laurent/intermediates/sim/series_at_obs_locations/{key}.nc')

In [7]:
key = 'tas_T'
series[key].to_netcdf(f'/runoff/gulley/St_Laurent/intermediates/sim/series_at_obs_locations/{key}.nc')